# VERGIL — Production GRPO Training Notebook

Trains **Qwen2.5-0.5B** to act as the VERGIL commitment-management agent using **Group Relative Policy Optimization (GRPO)** with Unsloth 4-bit quantization.

## What this notebook does (in order)
1. **Setup** — install deps, clone repo
2. **Environment sanity check** — verify VERGIL CDG engine works
3. **Pre-training baseline** — record what the base model does BEFORE training
4. **Load model with LoRA** — rank=64, alpha=128 for reasoning capacity
5. **Generate training dataset** — 500+ prompts across all 4 curriculum stages
6. **GRPO training** — 3 epochs, proper group-relative advantage estimation
7. **Post-training evaluation** — measure improvement vs baseline
8. **Inference demo** — show the model's `<think>` reasoning visible
9. **Save + push to HuggingFace**

## Hardware
**Required:** T4 GPU (free Colab tier)  
**Estimated time:** 2–3 hours total

---
## Section 1 — Setup
Run this once at the start of every Colab session.

In [ ]:
# ── Cell 1: Install all dependencies ─────────────────────────────────────
# unsloth: 4-bit quantized fine-tuning (fits on T4)
# trl: HuggingFace alignment library (contains GRPOTrainer)
# peft: LoRA adapter framework
# gymnasium + networkx: VERGIL environment runtime

!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "xformers<0.0.27" "trl>=0.9.0" peft accelerate bitsandbytes
!pip install -q gymnasium networkx numpy scipy datasets huggingface_hub

print("✅ All dependencies installed.")

In [ ]:
# ── Cell 2: Clone VERGIL repo and set Python path ────────────────────────
import os, sys

if not os.path.exists('Vergil'):
    !git clone https://github.com/laksh718/Vergil.git
else:
    print("Repo already cloned — pulling latest changes...")
    !cd Vergil && git pull origin main

%cd Vergil
sys.path.insert(0, '.')

# Verify key files are present
import os
required = [
    'vergil/core/env.py',
    'vergil/core/reward.py',
    'scripts/train_grpo_colab.py',
    'scenarios/scenario_01_simple.json',
    'scenarios/scenario_11_impossible_math.json',
]
for f in required:
    status = '✅' if os.path.exists(f) else '❌ MISSING'
    print(f"  {status}  {f}")

print("\n✅ Repository ready.")

---
## Section 2 — VERGIL Environment Initialization & Sanity Check
Before touching the LLM, verify the CDG engine works end-to-end.

In [ ]:
# ── Cell 3: All imports ───────────────────────────────────────────────────
import json, copy, time
import numpy as np
from datetime import datetime, timedelta
from pathlib import Path

from vergil.core.env import VERGILEnv
from vergil.core.pomdp import POMDPWrapper
from vergil.core.types import AgentAction, ActionType, CommitmentStatus
from vergil.curriculum.scenario_generator import ScenarioGenerator
from vergil.curriculum.curriculum_engine import CurriculumEngine
from vergil.curriculum.failure_db import FailureTopologyDatabase
from scripts.train_grpo_colab import state_to_prompt, parse_llm_output, simulate_task_progress

print("✅ All VERGIL modules imported.")

In [ ]:
# ── Cell 4: Environment sanity check ─────────────────────────────────────
# Boots VERGIL, loads scenario 01, takes 3 actions, confirms CDG + reward work.

env_test = VERGILEnv(seed=42, config={'max_steps_per_episode': 20, 'step_hours': 2})
pomdp_test = POMDPWrapper(env_test)

with open('scenarios/scenario_01_simple.json') as f:
    sc_test = json.load(f)

state, belief, info = pomdp_test.reset(scenario=sc_test)

print("=" * 55)
print(f"  VERGIL Environment Sanity Check")
print("=" * 55)
print(f"  Nodes loaded  : {len(state.cdg_nodes)}")
print(f"  SAT score     : {state.satisfiability_score:.2f}")
print(f"  Avail hours   : {state.available_hours_next_48h:.1f}h")
print(f"  Trust network : {list(state.trust_entries.keys())}")

# Take 3 test actions
rewards_so_far = []
for i, atype in enumerate([ActionType.ACCEPT, ActionType.DO_NOTHING, ActionType.DO_NOTHING]):
    pending = [n for n in state.cdg_nodes if n.status == CommitmentStatus.PENDING]
    action = AgentAction(
        action_type=atype,
        target_node_id=pending[0].node_id if pending and atype != ActionType.DO_NOTHING else None,
    )
    state, belief, reward, term, trunc, step_info = pomdp_test.step(action)
    rewards_so_far.append(reward)
    print(f"  Step {i+1}: {atype.value:15s}  reward={reward:+.4f}  SAT={state.satisfiability_score:.2f}")

print("=" * 55)
print(f"  ✅ CDG engine working correctly.")

# Also verify the chain-of-thought prompt format
sample_prompt = state_to_prompt(state, env_test)
has_think = '<think>' in sample_prompt
print(f"  ✅ Prompt has <think> block: {has_think}")
print(f"  Prompt length: {len(sample_prompt)} chars")

del env_test, pomdp_test, sc_test

---
## Section 3 — Pre-Training Baseline
Load the **raw base model** (no LoRA) and run 15 evaluation episodes.  
This records what the model does BEFORE any training — we need this to prove improvement later.  

**Expected baseline metrics:** reward ≈ 0.10–0.30, fulfillment ≈ 40–60% (mostly random/greedy)

In [ ]:
# ── Cell 5: Load BASE model for baseline evaluation ───────────────────────
# Load WITHOUT LoRA — this is the unmodified Qwen2.5-0.5B
# We use it only for inference (not training), so no LoRA needed yet.

from unsloth import FastLanguageModel

print("📦 Loading base model (no LoRA) for baseline...")
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-0.5B-Instruct",
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,
)
FastLanguageModel.for_inference(base_model)  # Enable inference mode

print(f"  ✅ Base model loaded.")
print(f"  Total params: {base_model.num_parameters():,}")

In [ ]:
# ── Cell 6: Run baseline evaluation (15 episodes, 4 curriculum stages) ───
import torch

def run_eval_episodes(model, tokenizer, n_episodes=15, label="Model"):
    """
    Run N evaluation episodes and return a metrics dict.
    Covers stages 1–4 (3-4 episodes each) to get a complete picture.
    """
    eval_env = VERGILEnv(seed=77, config={'max_steps_per_episode': 20, 'step_hours': 2})
    eval_pomdp = POMDPWrapper(eval_env)
    eval_failure_db = FailureTopologyDatabase(db_path='/tmp/vergil_eval_ftd.sqlite')
    eval_scenario_gen = ScenarioGenerator(seed=77)
    eval_curriculum = CurriculumEngine(
        failure_db=eval_failure_db, scenario_generator=eval_scenario_gen, initial_stage=1
    )

    all_rewards, all_fulfillments, all_trust_finals = [], [], []
    decision_counts = {'accept': 0, 'decline': 0, 'counter_propose': 0, 'do_nothing': 0}

    for ep in range(n_episodes):
        # Cycle through stages 1-4
        stage = (ep % 4) + 1
        eval_env.curriculum_stage = stage
        eval_curriculum.current_stage = stage
        scenario = eval_curriculum.generate_next_episode()
        state, belief, _ = eval_pomdp.reset(scenario=scenario)

        ep_reward = 0.0
        for step in range(eval_env._max_steps):
            simulate_task_progress(eval_env)
            prompt = state_to_prompt(state, eval_env)

            # Run model inference
            inputs = tokenizer(
                prompt, return_tensors="pt",
                truncation=True, max_length=1800
            ).to(model.device)

            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=400,
                    temperature=0.1,   # Low temp for eval — deterministic decisions
                    do_sample=False,
                    pad_token_id=tokenizer.eos_token_id,
                )

            completion = tokenizer.decode(
                outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True
            )

            pending = [n for n in state.cdg_nodes if n.status == CommitmentStatus.PENDING]
            action_type, target = parse_llm_output(completion, pending)

            if action_type in (ActionType.ACCEPT, ActionType.DECLINE, ActionType.COUNTER_PROPOSE):
                if not pending:
                    action_type, target = ActionType.DO_NOTHING, None
                elif target is None:
                    target = pending[0].node_id

            decision_counts[action_type.value] = decision_counts.get(action_type.value, 0) + 1
            action = AgentAction(action_type=action_type, target_node_id=target)
            state, belief, reward, terminated, truncated, _ = eval_pomdp.step(action)
            simulate_task_progress(eval_env)
            ep_reward += reward
            if terminated or truncated:
                break

        # Episode metrics
        n_completed = sum(1 for n in state.cdg_nodes if n.status == CommitmentStatus.COMPLETED)
        n_accepted  = sum(1 for n in state.cdg_nodes
                         if n.status in (CommitmentStatus.ACCEPTED, CommitmentStatus.COMPLETED))
        fulfillment = n_completed / max(1, n_accepted)
        trust_avg   = np.mean([te.trust_score for te in state.trust_entries.values()])

        all_rewards.append(ep_reward)
        all_fulfillments.append(fulfillment)
        all_trust_finals.append(trust_avg)

    total_decisions = sum(decision_counts.values())
    print(f"\n{'='*55}")
    print(f"  {label} — Evaluation Results ({n_episodes} episodes)")
    print(f"{'='*55}")
    print(f"  Mean reward       : {np.mean(all_rewards):+.4f}  (σ={np.std(all_rewards):.4f})")
    print(f"  Fulfillment rate  : {np.mean(all_fulfillments):.1%}")
    print(f"  Avg final trust   : {np.mean(all_trust_finals):.3f}")
    print(f"  Decision split    : accept={decision_counts.get('accept',0)/max(1,total_decisions):.0%} "
          f"decline={decision_counts.get('decline',0)/max(1,total_decisions):.0%} "
          f"counter={decision_counts.get('counter_propose',0)/max(1,total_decisions):.0%} "
          f"wait={decision_counts.get('do_nothing',0)/max(1,total_decisions):.0%}")
    print(f"{'='*55}")

    return {
        'mean_reward': float(np.mean(all_rewards)),
        'std_reward': float(np.std(all_rewards)),
        'fulfillment_rate': float(np.mean(all_fulfillments)),
        'avg_trust': float(np.mean(all_trust_finals)),
        'decision_counts': decision_counts,
    }

# Run baseline
BASELINE_METRICS = run_eval_episodes(base_model, base_tokenizer, n_episodes=15, label="BASE MODEL (pre-training)")

# Free base model from memory before loading LoRA version
del base_model, base_tokenizer
import gc; gc.collect()
torch.cuda.empty_cache()
print("\n📋 Baseline saved to BASELINE_METRICS. Memory freed for LoRA model.")

---
## Section 4 — Load Model with LoRA (Training Configuration)
Now load the model WITH LoRA adapters for training.  
**rank=64, alpha=128** — significantly more expressive than the default rank=16.

In [ ]:
# ── Cell 7: Load Qwen2.5-0.5B with LoRA rank=64 ───────────────────────────
from unsloth import FastLanguageModel
import torch

print("📦 Loading model with LoRA (rank=64)...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-0.5B-Instruct",
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=64,                 # Rank — higher = more reasoning capacity
    lora_alpha=128,       # alpha = 2×r is optimal for most tasks
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",   # Attention
        "gate_proj", "up_proj", "down_proj",        # FFN
    ],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

trainable = model.num_parameters(only_trainable=True)
total     = model.num_parameters()
print(f"\n  Trainable params : {trainable:>12,}  ({trainable/total:.2%} of total)")
print(f"  Total params     : {total:>12,}")
print(f"  GPU memory       : {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print("  ✅ LoRA model ready for training.")

---
## Section 5 — Generate Training Dataset
Generate **500 diverse prompts** across all 4 curriculum stages with a balanced action mix.  
Stage 1–2: foundation + complexity.  Stage 3–4: social dynamics + adversarial.

**Action mix:** 40% accept, 20% decline, 20% counter, 20% wait → creates states for every decision type.

In [ ]:
# ── Cell 8: Generate 500 training prompts across all curriculum stages ────
from datasets import Dataset

gen_env = VERGILEnv(seed=42, config={'max_steps_per_episode': 20, 'step_hours': 2})
gen_pomdp = POMDPWrapper(gen_env)
gen_failure_db = FailureTopologyDatabase(db_path='/tmp/vergil_ftd_grpo.sqlite')
gen_scenario_gen = ScenarioGenerator(seed=42)
gen_curriculum = CurriculumEngine(
    failure_db=gen_failure_db, scenario_generator=gen_scenario_gen, initial_stage=1
)

training_prompts = []

# Stage distribution: 80 + 120 + 150 + 150 = 500 episodes
STAGE_EPISODES = {1: 80, 2: 120, 3: 150, 4: 150}

# Per-step action distribution for diversity
def sample_action(state, rng):
    pending = [n for n in state.cdg_nodes if n.status == CommitmentStatus.PENDING]
    roll = rng.random()
    if pending and roll < 0.40:
        return AgentAction(ActionType.ACCEPT, target_node_id=pending[0].node_id)
    elif pending and roll < 0.60:
        return AgentAction(ActionType.DECLINE, target_node_id=pending[0].node_id)
    elif pending and roll < 0.80:
        return AgentAction(ActionType.COUNTER_PROPOSE, target_node_id=pending[0].node_id)
    return AgentAction(ActionType.DO_NOTHING)

rng = np.random.default_rng(seed=42)

for stage, n_episodes in STAGE_EPISODES.items():
    gen_env.curriculum_stage = stage
    gen_curriculum.current_stage = stage
    stage_prompts = 0

    for ep in range(n_episodes):
        scenario = gen_curriculum.generate_next_episode()
        state, belief, _ = gen_pomdp.reset(scenario=scenario)

        for step in range(min(8, gen_env._max_steps)):
            simulate_task_progress(gen_env)
            prompt = state_to_prompt(state, gen_env)
            training_prompts.append(prompt)
            stage_prompts += 1

            action = sample_action(state, rng)
            state, belief, _, term, trunc, _ = gen_pomdp.step(action)
            simulate_task_progress(gen_env)
            if term or trunc:
                break

    print(f"  Stage {stage}: {n_episodes} episodes → {stage_prompts} prompts")

# Shuffle so stages are interleaved (prevents curriculum-order bias)
rng.shuffle(training_prompts)

dataset = Dataset.from_dict({"prompt": training_prompts})

print(f"\n  Total prompts      : {len(dataset)}")
print(f"  <think> format     : {'✅ present' if '<think>' in training_prompts[0] else '❌ missing'}")
print(f"  Sample prompt len  : {len(training_prompts[0])} chars")
print("  ✅ Training dataset ready.")

# Preview a sample prompt
print("\n" + "─"*55)
print("SAMPLE PROMPT PREVIEW (first 800 chars):")
print("─"*55)
print(training_prompts[0][:800])
print("[... truncated ...]")

del gen_env, gen_pomdp, gen_failure_db, gen_scenario_gen, gen_curriculum

---
## Section 6 — GRPO Training
**Key design decisions:**
- `num_generations=8` — 8 rollouts per CDG topology for stable group advantage
- `max_completion_length=512` — enough room for `<think>...</think>` + JSON
- Each group of 8 completions evaluates from the **same** VERGIL episode state (env snapshot/restore)
- 3 epochs over the shuffled dataset
- Reward has format bonus: +0.03 for valid JSON, +0.02 for `<think>` block

In [ ]:
# ── Cell 9: Define the GRPO reward function ───────────────────────────────
# This is the most critical cell — it bridges TRL with the VERGIL environment.
#
# CORRECTNESS FIX: GRPO generates N completions from the same prompt.
# All N completions MUST be evaluated from the SAME starting env state.
# We create one fresh VERGIL episode per group and snapshot/restore it.

import json as _json

NUM_GENERATIONS = 8  # Must match GRPOConfig.num_generations

# Fresh infrastructure for reward evaluation (isolated from dataset generation)
reward_env = VERGILEnv(seed=99, config={'max_steps_per_episode': 20, 'step_hours': 2})
reward_pomdp = POMDPWrapper(reward_env)
reward_failure_db = FailureTopologyDatabase(db_path='/tmp/vergil_reward_ftd.sqlite')
reward_scenario_gen = ScenarioGenerator(seed=99)
reward_curriculum = CurriculumEngine(
    failure_db=reward_failure_db, scenario_generator=reward_scenario_gen, initial_stage=1
)

# Global group counter — used to gradually increase scenario difficulty
_group_counter = 0


def vergil_reward_fn(prompts, completions, **kw):
    """
    TRL GRPOTrainer reward function.

    For each group of NUM_GENERATIONS completions:
      1. Create a fresh VERGIL episode at the appropriate curriculum stage
      2. Snapshot the initial state
      3. For each completion: restore snapshot → parse action → step env → get reward
      4. Add format quality bonus (JSON validity + <think> block presence)

    Returns: list of float rewards, one per completion
    """
    global _group_counter
    rewards = []

    for group_start in range(0, len(prompts), NUM_GENERATIONS):
        group_completions = completions[group_start : group_start + NUM_GENERATIONS]
        _group_counter += 1

        # Curriculum stage: start easy, ramp up over training
        # Groups 0–50: stage 1–2, Groups 50–150: stage 2–3, Groups 150+: stage 3–4
        if _group_counter < 50:
            stage = 1 if _group_counter < 25 else 2
        elif _group_counter < 150:
            stage = 2 if _group_counter < 100 else 3
        else:
            stage = 3 if _group_counter % 5 != 0 else 4  # 20% stage 4

        # ── Create fresh episode for this group ──────────────────────────
        reward_env.curriculum_stage = stage
        reward_curriculum.current_stage = stage
        scenario = reward_curriculum.generate_next_episode()
        state, _, _ = reward_pomdp.reset(scenario=scenario)

        # ── Snapshot state (all 8 completions evaluate from here) ────────
        snap = {
            'state':      copy.deepcopy(reward_env._state),
            'hidden':     copy.deepcopy(reward_env._hidden),
            'cdg_nodes':  copy.deepcopy(reward_env.cdg._nodes),
            'step_count': reward_env._step_count,
            'md_trust':   copy.deepcopy(getattr(reward_env, 'multidim_trust', {})),
        }

        for completion in group_completions:
            # ── Restore to same starting state ───────────────────────────
            reward_env._state        = copy.deepcopy(snap['state'])
            reward_env._hidden       = copy.deepcopy(snap['hidden'])
            reward_env.cdg._nodes    = copy.deepcopy(snap['cdg_nodes'])
            reward_env._step_count   = snap['step_count']
            reward_env.multidim_trust = copy.deepcopy(snap['md_trust'])

            try:
                s = reward_env._state
                pending = [n for n in s.cdg_nodes if n.status == CommitmentStatus.PENDING]

                action_type, target = parse_llm_output(completion, pending)

                if action_type in (ActionType.ACCEPT, ActionType.DECLINE, ActionType.COUNTER_PROPOSE):
                    if not pending:
                        action_type, target = ActionType.DO_NOTHING, None
                    elif target is None:
                        target = pending[0].node_id

                # Feasibility prediction from capacity math
                available  = getattr(s, 'available_hours_next_48h', 8.0)
                committed  = sum(n.estimated_duration_hours for n in s.cdg_nodes
                                if n.status.value in ('accepted', 'in_progress'))
                t_node     = next((n for n in s.cdg_nodes if n.node_id == target), None)
                new_cost   = t_node.estimated_duration_hours if t_node else 0.0
                feas_pred  = float(committed + new_cost <= available)

                action = AgentAction(
                    action_type=action_type,
                    target_node_id=target,
                    feasibility_prediction=feas_pred,
                )
                if action_type == ActionType.COUNTER_PROPOSE and t_node:
                    action.proposed_deadline = s.current_time + timedelta(
                        hours=t_node.estimated_duration_hours * 1.5)

                simulate_task_progress(reward_env)
                _, _, reward, _, _, _ = reward_pomdp.step(action)
                simulate_task_progress(reward_env)

                # ── Format quality bonus ──────────────────────────────────
                has_json  = '{' in completion and '}' in completion
                has_think = '<think>' in completion and '</think>' in completion

                # Check if JSON has all required keys
                try:
                    s_idx = completion.find('{')
                    e_idx = completion.rfind('}') + 1
                    parsed = _json.loads(completion[s_idx:e_idx]) if s_idx >= 0 else {}
                    has_all_keys = all(k in parsed for k in ('action', 'target', 'reasoning'))
                except Exception:
                    has_all_keys = False

                fmt_bonus  = 0.03 if (has_json and has_all_keys) else (0.01 if has_json else -0.05)
                fmt_bonus += 0.02 if has_think else 0.0

                rewards.append(float(reward + fmt_bonus))

            except Exception as ex:
                rewards.append(-0.10)  # Penalty for crashes

    return rewards


# ── Quick reward function smoke test ─────────────────────────────────────
test_prompts = ["test"] * NUM_GENERATIONS
test_completions = ['{"action": "accept", "target": "C1", "reasoning": "ok"}'] * NUM_GENERATIONS
test_rewards = vergil_reward_fn(test_prompts, test_completions)
print(f"✅ Reward function smoke test passed.")
print(f"   {NUM_GENERATIONS} completions → {len(test_rewards)} rewards")
print(f"   Sample rewards: {[round(r,4) for r in test_rewards[:4]]}")
_group_counter = 0  # Reset counter after smoke test

In [ ]:
# ── Cell 10: GRPO Training ────────────────────────────────────────────────
from trl import GRPOConfig, GRPOTrainer
import time

print("🚀 Starting GRPO training...")
print(f"   Dataset size     : {len(dataset)} prompts")
print(f"   LoRA rank        : 64  (alpha=128)")
print(f"   num_generations  : {NUM_GENERATIONS}")
print(f"   Epochs           : 3")
print(f"   Max completion   : 512 tokens (for <think> blocks)")
print(f"   Learning rate    : 2e-5")
print("")

training_config = GRPOConfig(
    output_dir="/tmp/vergil_grpo_output",

    # ── Epochs & batch ────────────────────────────────────────────────
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,   # Effective batch = 16

    # ── Learning rate ────────────────────────────────────────────────
    learning_rate=2e-5,
    warmup_steps=30,
    lr_scheduler_type="cosine",      # Cosine decay is smoother for GRPO

    # ── GRPO generation ──────────────────────────────────────────────
    num_generations=NUM_GENERATIONS,
    max_completion_length=512,       # Must fit <think>...</think> + JSON
    temperature=0.9,                 # Exploration during rollouts
    top_p=0.95,

    # ── Logging ──────────────────────────────────────────────────────
    logging_steps=5,
    save_steps=100,
    report_to="none",

    # ── Gradient clipping (prevents instability with large LoRA) ─────
    max_grad_norm=1.0,
)

trainer = GRPOTrainer(
    model=model,
    args=training_config,
    train_dataset=dataset,
    reward_funcs=[vergil_reward_fn],
    processing_class=tokenizer,
)

# ── Train ──────────────────────────────────────────────────────────────
t0 = time.time()
train_result = trainer.train()
elapsed = time.time() - t0

print(f"\n{'='*55}")
print(f"  ✅ GRPO Training Complete")
print(f"  Training time  : {elapsed/60:.1f} minutes")
print(f"  Final loss     : {train_result.training_loss:.6f}")
print(f"  Total steps    : {train_result.global_step}")
print(f"{'='*55}")

# Enable inference mode for evaluation
FastLanguageModel.for_inference(model)

---
## Section 7 — Post-Training Evaluation
Run 15 evaluation episodes with the **trained** model and compare against the baseline.

Expected improvements over baseline:
- Reward: +0.3 to +0.8
- Fulfillment rate: +15–30%
- Decision split: more balanced (less always-accept)

In [ ]:
# ── Cell 11: Post-training evaluation ─────────────────────────────────────
print("📊 Running post-training evaluation...")
POST_METRICS = run_eval_episodes(model, tokenizer, n_episodes=15, label="VERGIL-TRAINED")

In [ ]:
# ── Cell 12: Before / After comparison table ──────────────────────────────
b = BASELINE_METRICS
p = POST_METRICS

def delta_str(before, after, higher_is_better=True, pct=False):
    d = after - before
    sign = '+' if d >= 0 else ''
    fmt = f"{sign}{d:.1%}" if pct else f"{sign}{d:.4f}"
    arrow = '↑' if (d > 0) == higher_is_better else '↓'
    return f"{fmt}  {arrow}"

print("\n" + "="*65)
print(f"  {'METRIC':<28} {'BEFORE':>10}   {'AFTER':>10}   {'DELTA':>14}")
print("="*65)
print(f"  {'Mean episode reward':<28} {b['mean_reward']:>+10.4f}   {p['mean_reward']:>+10.4f}   {delta_str(b['mean_reward'], p['mean_reward']):>14}")
print(f"  {'Reward std dev':<28} {b['std_reward']:>10.4f}   {p['std_reward']:>10.4f}   {delta_str(b['std_reward'], p['std_reward'], higher_is_better=False):>14}")
print(f"  {'Fulfillment rate':<28} {b['fulfillment_rate']:>10.1%}   {p['fulfillment_rate']:>10.1%}   {delta_str(b['fulfillment_rate'], p['fulfillment_rate'], pct=True):>14}")
print(f"  {'Average final trust':<28} {b['avg_trust']:>10.3f}   {p['avg_trust']:>10.3f}   {delta_str(b['avg_trust'], p['avg_trust']):>14}")
print("="*65)

# Decision distribution shift
print("\n  Decision distribution shift:")
bd, pd = b['decision_counts'], p['decision_counts']
bt = max(1, sum(bd.values()))
pt = max(1, sum(pd.values()))
for action in ['accept', 'decline', 'counter_propose', 'do_nothing']:
    bpct = bd.get(action, 0) / bt
    ppct = pd.get(action, 0) / pt
    bar = '█' * int(ppct * 20)
    print(f"  {action:<18} before={bpct:.0%}  after={ppct:.0%}  {bar}")

# Save comparison to JSON for the API dashboard
comparison = {
    'baseline': b,
    'post_training': p,
    'improvement': {
        'reward_delta': p['mean_reward'] - b['mean_reward'],
        'fulfillment_delta': p['fulfillment_rate'] - b['fulfillment_rate'],
        'trust_delta': p['avg_trust'] - b['avg_trust'],
    }
}
Path('/tmp/vergil_grpo_output').mkdir(exist_ok=True)
with open('/tmp/vergil_grpo_output/comparison.json', 'w') as f:
    json.dump(comparison, f, indent=2)
print("\n  ✅ Comparison saved to /tmp/vergil_grpo_output/comparison.json")

---
## Section 8 — Inference Demo
Run the trained model on a hard scenario and show its full `<think>` reasoning chain.  
This is the "smoking gun" — proves the model is doing CDG reasoning, not random guessing.

In [ ]:
# ── Cell 13: Inference demo — show <think> reasoning on hard scenario ─────
import textwrap

print("🎬 Running inference demo on adversarial scenarios...\n")

demo_env = VERGILEnv(seed=1234, config={'max_steps_per_episode': 20, 'step_hours': 2})
demo_pomdp = POMDPWrapper(demo_env)
demo_scenario_gen = ScenarioGenerator(seed=1234)

# Test on 3 different difficulty levels
demo_scenarios = [
    ('scenarios/scenario_07_simultaneous_infeasibility.json',
     'Stage 4: Simultaneous Triple Infeasibility'),
    ('scenarios/scenario_11_impossible_math.json',
     'Stage 4: Impossible Math (must decline)'),
    ('scenarios/scenario_10_deadline_cascade.json',
     'Stage 4: Deadline Cascade Chain'),
]

for scenario_path, scenario_name in demo_scenarios:
    print(f"{'='*65}")
    print(f"  SCENARIO: {scenario_name}")
    print(f"{'='*65}")

    with open(scenario_path) as f:
        sc = json.load(f)

    state, belief, _ = demo_pomdp.reset(scenario=sc)
    prompt = state_to_prompt(state, demo_env)

    inputs = tokenizer(
        prompt, return_tensors="pt",
        truncation=True, max_length=1800
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.3,   # Slightly warmer for demo variety
            do_sample=True,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id,
        )

    completion = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True
    )

    # Parse the action
    pending = [n for n in state.cdg_nodes if n.status == CommitmentStatus.PENDING]
    action_type, target = parse_llm_output(completion, pending)

    # Display the reasoning
    print(f"\n  Pending commitments: {[n.node_id for n in pending]}")
    print(f"  Available hours    : {state.available_hours_next_48h:.1f}h")
    print(f"  SAT score          : {state.satisfiability_score:.2f}")
    print()

    # Extract and display <think> block
    if '<think>' in completion and '</think>' in completion:
        think_start = completion.find('<think>') + len('<think>')
        think_end = completion.find('</think>')
        think_content = completion[think_start:think_end].strip()
        print("  🧠 AGENT REASONING:")
        for line in think_content.split('\n'):
            if line.strip():
                print(f"     {line.strip()}")
    else:
        print("  [No <think> block — model didn't reason in structured format]")

    # Extract and display final JSON decision
    try:
        s_idx = completion.rfind('{')
        e_idx = completion.rfind('}') + 1
        decision = json.loads(completion[s_idx:e_idx]) if s_idx >= 0 else {}
        print(f"\n  ⚡ DECISION: {decision.get('action', '?').upper()} → {decision.get('target', 'none')}")
        print(f"  📝 REASON: {decision.get('reasoning', 'none')}")
    except Exception:
        print(f"\n  ⚡ PARSED ACTION: {action_type.value.upper()} → {target}")

    print()

del demo_env, demo_pomdp

---
## Section 9 — Save Model + Push to HuggingFace
Save the LoRA adapter weights locally and push to your HF Hub repo.  
**Replace `YOUR_HF_USERNAME` before running.**

In [ ]:
# ── Cell 14: Save model locally ───────────────────────────────────────────
SAVE_PATH = "/tmp/vergil_grpo_model"

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

# Save the comparison metrics alongside the model
import shutil
shutil.copy('/tmp/vergil_grpo_output/comparison.json', f'{SAVE_PATH}/training_comparison.json')

# Verify saved files
saved_files = list(Path(SAVE_PATH).glob('*'))
print(f"Model saved to: {SAVE_PATH}")
print(f"Saved files:")
for f in sorted(saved_files):
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name:<40} {size_mb:>8.2f} MB")

print("\n✅ Model saved locally.")

In [ ]:
# ── Cell 15: Push to HuggingFace Hub ──────────────────────────────────────
# IMPORTANT: Replace YOUR_HF_USERNAME with your actual HuggingFace username

from huggingface_hub import notebook_login
notebook_login()  # Opens login widget — paste your HF write token

In [ ]:
# ── Cell 16: Push model + training metadata to HF ─────────────────────────
HF_REPO_NAME = "YOUR_HF_USERNAME/vergil-commitment-engine"  # ← CHANGE THIS

# Push LoRA weights + tokenizer
model.push_to_hub(HF_REPO_NAME, commit_message="VERGIL GRPO training — rank=64, 3 epochs")
tokenizer.push_to_hub(HF_REPO_NAME)

# Push training comparison metadata
from huggingface_hub import HfApi
api = HfApi()
api.upload_file(
    path_or_fileobj=f"{SAVE_PATH}/training_comparison.json",
    path_in_repo="training_comparison.json",
    repo_id=HF_REPO_NAME,
    commit_message="Add before/after training metrics",
)

print(f"\n✅ Model pushed to: https://huggingface.co/{HF_REPO_NAME}")
print(f"\n{'='*55}")
print(f"  TRAINING SUMMARY")
print(f"{'='*55}")
print(f"  Model           : {HF_REPO_NAME}")
print(f"  LoRA rank       : 64  (alpha=128)")
print(f"  GRPO groups     : {NUM_GENERATIONS} completions each")
print(f"  Dataset         : {len(dataset)} prompts (stages 1–4)")
print(f"  Reward Δ        : {POST_METRICS['mean_reward'] - BASELINE_METRICS['mean_reward']:+.4f}")
print(f"  Fulfillment Δ   : {POST_METRICS['fulfillment_rate'] - BASELINE_METRICS['fulfillment_rate']:+.1%}")
print(f"  Trust Δ         : {POST_METRICS['avg_trust'] - BASELINE_METRICS['avg_trust']:+.4f}")
print(f"{'='*55}")